<a href="https://colab.research.google.com/github/mpopovic10/cookify/blob/main/Copy_of_ingredient_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
import os


drive.mount('/content/drive')


path = "/content/drive/MyDrive/cookify/Food Ingredients and Recipe Dataset with Image Name Mapping.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import pandas as pd

df = pd.read_csv(path)
df['Title'] = df['Title'].fillna('Untitled')
df['Instructions'] = df['Instructions'].fillna('')
display(df.head())
display(df.columns)
display(df.info())

,Unnamed: 0,Title,Ingredients,Instructions,Image_Name,Cleaned_Ingredients
0,0,Miso-Butter Roast Chicken With Acorn Squash Pa...,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...","Pat chicken dry with paper towels, season all ...",miso-butter-roast-chicken-acorn-squash-panzanella,"['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher..."
1,1,Crispy Salt and Pepper Potatoes,"['2 large egg whites', '1 pound new potatoes (...",Preheat oven to 400°F and line a rimmed baking...,crispy-salt-and-pepper-potatoes-dan-kluger,"['2 large egg whites', '1 pound new potatoes (..."
2,2,Thanksgiving Mac and Cheese,"['1 cup evaporated milk', '1 cup whole milk', ...",Place a rack in middle of oven; preheat to 400...,thanksgiving-mac-and-cheese-erick-williams,"['1 cup evaporated milk', '1 cup whole milk', ..."
3,3,Italian Sausage and Bread Stuffing,"['1 (¾- to 1-pound) round Italian loaf, cut in...",Preheat oven to 350°F with rack in middle. Gen...,italian-sausage-and-bread-stuffing-240559,"['1 (¾- to 1-pound) round Italian loaf, cut in..."
4,4,Newton's Law,"['1 teaspoon dark brown sugar', '1 teaspoon ho...",Stir together brown sugar and hot water in a c...,newtons-law-apple-bourbon-cocktail,"['1 teaspoon dark brown sugar', '1 teaspoon ho..."


Index(['Unnamed: 0', 'Title', 'Ingredients', 'Instructions', 'Image_Name',
       'Cleaned_Ingredients'],
      dtype='object')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13501 entries, 0 to 13500
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Unnamed: 0           13501 non-null  int64 
 1   Title                13501 non-null  object
 2   Ingredients          13501 non-null  object
 3   Instructions         13501 non-null  object
 4   Image_Name           13501 non-null  object
 5   Cleaned_Ingredients  13501 non-null  object
dtypes: int64(1), object(5)
memory usage: 633.0+ KB


None

In [9]:
import re

def clean_ingredient(text):
  text = text.lower()

  text = re.sub(r'\d+\/\d+|\d+', '', text)
  text = re.sub(r'[¼½¾⅓⅔⅛]', '', text)

  units = ['cup', 'cups', 'tbsp', 'tsp', 'tablespoon', 'teaspoon',
           'pound', 'lb', 'oz', 'ounce', 'grams', 'kg', 'ml']

  for u in units:
    text = text.replace(u, '')

  text = re.sub(r'[^a-z\s]', '', text)

  text = re.sub(r'\s+', ' ', text).strip()

  return text

In [10]:
def clean_ingredient_list(ingredient_list):
  cleaned = []

  for i in ingredient_list:
    cleaned_item = clean_ingredient(i)
    cleaned.append(cleaned_item)

  return cleaned

In [11]:
type(df["Cleaned_Ingredients"].iloc[0])

str

In [12]:
import ast

df["Cleaned_Ingredients"] = df["Cleaned_Ingredients"].apply(ast.literal_eval)

In [13]:
df['ingredients_clean'] = df['Cleaned_Ingredients'].apply(clean_ingredient_list)

In [14]:
df['ingredients_clean'].head()

,ingredients_clean
0,"[whole chicken, kosher salt divided plus more,..."
1,"[large egg whites, new potatoes about inch in ..."
2,"[evaporated milk, whole milk, garlic powder, o..."
3,"[to round italian loaf cut into inch cubes s, ..."
4,"[dark brown sugar, hot water, bourbon, fresh l..."


In [15]:
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [16]:
def lemmatize_ingredients(all_recipes_list):
  new_big_list = []
  for recipe in all_recipes_list:
    clean_recipe = []
    for word in recipe:
      root_word = lemmatizer.lemmatize(word)
      clean_recipe.append(root_word)
    new_big_list.append(clean_recipe)

  return new_big_list

In [17]:
df['lemmatized_ingredients'] = lemmatize_ingredients(df['ingredients_clean'])

In [18]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [19]:

def remove_stop_words(recipe_list):
    final_output = []

    for recipe in recipe_list:
        clean_recipe = []
        for word in recipe:

            if word not in stop_words:
                clean_recipe.append(word)
        final_output.append(clean_recipe)

    return final_output

df['final_ingredients'] = remove_stop_words(df['lemmatized_ingredients'])


In [20]:
df['final_ingredients'] = remove_stop_words(df['lemmatized_ingredients'])